## CPSC 390 Spring 26 Project 1 - Search
Name: Dennis Fomichev

### BFS, DFS, and A* search algorithms implementations


Romania roadmap encoded as dictionary of dictionaries (provided in the code template)

In [14]:
roadmap = dict(
    Arad=dict(Zerind=75, Sibiu=140, Timisoara=118),
    Bucharest=dict(Urziceni=85, Pitesti=101, Giurgiu=90, Fagaras=211),
    Craiova=dict(Drobeta=120, Rimnicu=146, Pitesti=138),
    Drobeta=dict(Craiova=120, Mehadia=75),
    Eforie=dict(Hirsova=86),
    Fagaras=dict(Bucharest=211, Sibiu=99),
    Giurgiu=dict(Bucharest=90),
    Hirsova=dict(Eforie=86, Urziceni=98),
    Iasi=dict(Vaslui=92, Neamt=87),
    Lugoj=dict(Timisoara=111, Mehadia=70),
    Mehadia=dict(Lugoj=70, Drobeta=75),
    Neamt=dict(Iasi=87),
    Oradea=dict(Zerind=71, Sibiu=151),
    Pitesti=dict(Bucharest=101, Rimnicu=97, Craiova=138),
    Rimnicu=dict(Craiova=146, Sibiu=80, Pitesti=97),
    Sibiu=dict(Arad=140, Fagaras=99, Oradea=151, Rimnicu=80),
    Timisoara=dict(Arad=118,Lugoj=111),
    Vaslui=dict(Iasi=92, Urziceni=98),
    Urziceni=dict(Vaslui=142, Bucharest=85, Hirsova=98),
    Zerind=dict(Arad=75, Oradea=71))

The straight-line distance to Bucharest, had to generate this dictionary based on the table provided

In [15]:
heuristic = dict(
    Arad=366, 
    Bucharest=0,
    Craiova=160,
    Drobeta=242,
    Eforie=161,
    Fagaras=176,
    Giurgiu=77,
    Hirsova=151,
    Iasi=226,
    Lugoj=244,
    Mehadia=241,
    Neamt=234,
    Oradea=380,
    Pitesti=100,
    Rimnicu=193,
    Sibiu=253,
    Timisoara=329,
    Urziceni=80,
    Vaslui=199,
    Zerind=374)

### Node object
Used in all 3 algorithms so that I can keep track of where we are in the tree, and it contains data like what city we are at, its parent, and path cost. 

In [16]:
class Node:
    # Set the default variables
    def __init__(self, city, parent, path_cost):
        self.city = city
        self.parent = parent
        self.path_cost = path_cost

    # Returns the path to the root/start city, starting from the current node/city
    def Path(self):
        node, path_back = self, []

        while node:
            path_back.append(node)
            node = node.parent

        return list(reversed(path_back))

    # Print out the node based only on parent, current city, and its overall path cost
    def FormatNormal(self):
      if self.parent:
        return f"{self.parent.city}->{self.city}({self.path_cost})"
      else:
        return f"{self.city}(0)"

    # Only for A*, do the same formatting but account for the heuristic costs in path cost
    def FormatHeuristic(self, heuristic):
      cost = self.path_cost + heuristic[self.city]

      if self.parent:
        return f"{self.parent.city}->{self.city}({self.path_cost}, {cost})"
      else:
        return f"{self.city}(0, {cost})"

### Breadth-First Search (BFS)

In [155]:
# Using FIFO data structure for BFS
from collections import deque

# Breadth-First Search function implementation that goes through the roadmap level by level using a deque to store the frontier
def BFS(roadmap, start, goal):
    startCity = Node(start, parent=None, path_cost=0) # Our starting city
    frontier = deque([startCity]) # Store all of the possible cities we haven't seen yet
    explored = []  # Cities that we've already seen, could be multiple paths to get to them

    # While we have cities to explore
    while frontier:
        # Get the next city
        currentNode = frontier.popleft()
        currentCity = currentNode.city

        # Skip if we already saw it
        if currentCity in explored:
            continue

        # Add it to the list of cities we've seen
        explored.append(currentCity)

        # If we are already at the goal, then we're done
        if currentCity == goal:
            print(f"Expanding {currentNode.FormatNormal()}")

            path = []

            for n in currentNode.Path():
                path.append(n.city)

            print("Goal found!")
            print(f"Solution path: {'->'.join(path)}")
            print(f"Total path cost: {currentNode.path_cost}")

            return # Don't run the function anymore, we are done!

        # Expand and add all the new nodes that we found to the end of frontier
        for newCity, cost in roadmap.get(currentCity).items():
            if newCity not in explored:
                newNode = Node(newCity, currentNode, currentNode.path_cost + cost)
                frontier.append(newNode)

        # Print out what was expanded right now and current frontier
        path = []

        print(f"Expanding {currentNode.FormatNormal()}")

        for n in frontier:
            path.append(n.FormatNormal())

        print(f"Frontier: [{', '.join(path)}]\n")

    # If there's nothing left to explore and we haven't found goal, there probably isn't a path to it
    print("No path found.")

Now, running this function with the call below for calculating the path between Arad and Bucharest should yield us a total path cost of 450, with the full path being Arad->Sibiu->Fagaras->Bucharest

In [157]:
BFS(roadmap, "Arad", "Bucharest")

Expanding Arad(0)
Frontier: [Arad->Zerind(75), Arad->Sibiu(140), Arad->Timisoara(118)]

Expanding Arad->Zerind(75)
Frontier: [Arad->Sibiu(140), Arad->Timisoara(118), Zerind->Oradea(146)]

Expanding Arad->Sibiu(140)
Frontier: [Arad->Timisoara(118), Zerind->Oradea(146), Sibiu->Fagaras(239), Sibiu->Oradea(291), Sibiu->Rimnicu(220)]

Expanding Arad->Timisoara(118)
Frontier: [Zerind->Oradea(146), Sibiu->Fagaras(239), Sibiu->Oradea(291), Sibiu->Rimnicu(220), Timisoara->Lugoj(229)]

Expanding Zerind->Oradea(146)
Frontier: [Sibiu->Fagaras(239), Sibiu->Oradea(291), Sibiu->Rimnicu(220), Timisoara->Lugoj(229)]

Expanding Sibiu->Fagaras(239)
Frontier: [Sibiu->Oradea(291), Sibiu->Rimnicu(220), Timisoara->Lugoj(229), Fagaras->Bucharest(450)]

Expanding Sibiu->Rimnicu(220)
Frontier: [Timisoara->Lugoj(229), Fagaras->Bucharest(450), Rimnicu->Craiova(366), Rimnicu->Pitesti(317)]

Expanding Timisoara->Lugoj(229)
Frontier: [Fagaras->Bucharest(450), Rimnicu->Craiova(366), Rimnicu->Pitesti(317), Lugoj->Meha

### Depth-First Search (DFS)

In [158]:
# Depth-First Search function implementation that goes through the roadmap depth-first using a list as a stack to store the frontier
def DFS(roadmap, start, goal):
    startCity = Node(start, parent=None, path_cost=0) # Our starting city
    frontier = [startCity] # Store all of the possible cities we haven't seen yet
    explored = []  # Cities that we've already seen, could be multiple paths to get to them

    # While we have cities to explore
    while frontier:
        # Get the next city
        currentNode = frontier.pop()
        currentCity = currentNode.city

        # Skip if we already saw it
        if currentCity in explored:
            continue

        # Add it to the list of cities we've seen
        explored.append(currentCity)

        # If we are already at the goal, then we're done
        if currentCity == goal:
            print(f"Expanding {currentNode.FormatNormal()}")

            path = []

            for n in currentNode.Path():
                path.append(n.city)

            print("Goal found!")
            print(f"Solution path: {'->'.join(path)}")
            print(f"Total path cost: {currentNode.path_cost}")

            return # Don't run the function anymore, we are done!

        # Expand and add all the new nodes
        for newCity, cost in sorted(roadmap.get(currentCity).items(), reverse=True):
            if newCity not in explored:
                newNode = Node(newCity, currentNode, currentNode.path_cost + cost)
                frontier.append(newNode)

        # Print out what was expanded right now and current frontier
        path = []

        print(f"Expanding {currentNode.FormatNormal()}")

        for n in frontier:
            path.append(n.FormatNormal())

        print(f"Frontier: [{', '.join(path)}]\n")

    # If there's nothing left to explore and we haven't found goal, there probably isn't a path to it
    print("No path found.")

Now, running this function with the call below for calculating the path between Arad and Bucharest should yield us a total path cost of 450, with the full path being Arad->Sibiu->Fagaras->Bucharest

In [159]:
DFS(roadmap, "Arad", "Bucharest")

Expanding Arad(0)
Frontier: [Arad->Zerind(75), Arad->Timisoara(118), Arad->Sibiu(140)]

Expanding Arad->Sibiu(140)
Frontier: [Arad->Zerind(75), Arad->Timisoara(118), Sibiu->Rimnicu(220), Sibiu->Oradea(291), Sibiu->Fagaras(239)]

Expanding Sibiu->Fagaras(239)
Frontier: [Arad->Zerind(75), Arad->Timisoara(118), Sibiu->Rimnicu(220), Sibiu->Oradea(291), Fagaras->Bucharest(450)]

Expanding Fagaras->Bucharest(450)
Goal found!
Solution path: Arad->Sibiu->Fagaras->Bucharest
Total path cost: 450


### A* Search

In [ ]:
# Using priority queue for A*
import heapq

# A* Search function implementation that uses f(n) = g(n) + h(n) to pick the best node to expand
def ASearch(roadmap, start, goal, heuristic):
    startCity = Node(start, parent=None, path_cost=0) # Our starting city
    frontier = [] # Store (f, city, node) so we expand by lowest f first
    f = 0 + heuristic.get(start) # f(n) = g(n) + h(n)
    heapq.heappush(frontier, (f, start, startCity))

    explored = []  # Cities that we've already seen, could be multiple paths to get to them
    nodeCosts = {start: 0} # Best known path cost to each city, default only contains the starting city

    # While we have cities to explore
    while frontier:
        # Get the next city (lowest f-value)
        currentF, currentCity, currentNode = heapq.heappop(frontier)

        # Skip if we already saw it
        if currentCity in explored:
            continue

        # Add it to the list of cities we've seen
        explored.append(currentCity)

        # If we are already at the goal, then we're done
        if currentCity == goal:
            print(f"Expanding {currentNode.FormatHeuristic(heuristic)}")

            path = []

            for n in currentNode.Path():
                path.append(n.city)

            print("Goal found!")
            print(f"Solution path: {'->'.join(path)}")
            print(f"Total path cost: {currentNode.path_cost}")

            return # Don't run the function anymore, we are done!

        # Expand and add all the new nodes that we found (if we found a better path to that city)
        for newCity, cost in roadmap.get(currentCity).items():
            newCost = currentNode.path_cost + cost

            # Add new path to frontier if there is lower cost
            if newCity not in nodeCosts or newCost < nodeCosts[newCity]:
                newNode = Node(newCity, currentNode, newCost)
                f = newCost + heuristic.get(newCity)

                heapq.heappush(frontier, (f, newCity, newNode))

                nodeCosts[newCity] = newCost

        # Print out what was expanded right now and current frontier
        path = []

        print(f"Expanding {currentNode.FormatHeuristic(heuristic)}")

        for (f, city, n) in frontier:
            path.append(n.FormatHeuristic(heuristic))

        print(f"Frontier: [{', '.join(path)}]\n")

    # If theres nothing left to explore and we haven't found goal, there probably isn't a path to it
    print("No path found.")

Now, running this function with the call below for calculating the path between Arad and Bucharest should yield us a total path cost of 418, with the full path being Arad->Sibiu->Rimnicu->Pitesti->Bucharest

In [149]:
ASearch(roadmap, "Arad", "Bucharest", heuristic)

Expanding Arad(0, 366)
Frontier: [Arad->Sibiu(140, 393), Arad->Zerind(75, 449), Arad->Timisoara(118, 447)]

Expanding Arad->Sibiu(140, 393)
Frontier: [Sibiu->Rimnicu(220, 413), Sibiu->Fagaras(239, 415), Arad->Timisoara(118, 447), Sibiu->Oradea(291, 671), Arad->Zerind(75, 449)]

Expanding Sibiu->Rimnicu(220, 413)
Frontier: [Sibiu->Fagaras(239, 415), Arad->Zerind(75, 449), Rimnicu->Pitesti(317, 417), Sibiu->Oradea(291, 671), Rimnicu->Craiova(366, 526), Arad->Timisoara(118, 447)]

Expanding Sibiu->Fagaras(239, 415)
Frontier: [Rimnicu->Pitesti(317, 417), Arad->Zerind(75, 449), Arad->Timisoara(118, 447), Sibiu->Oradea(291, 671), Rimnicu->Craiova(366, 526), Fagaras->Bucharest(450, 450)]

Expanding Rimnicu->Pitesti(317, 417)
Frontier: [Pitesti->Bucharest(418, 418), Arad->Zerind(75, 449), Arad->Timisoara(118, 447), Sibiu->Oradea(291, 671), Rimnicu->Craiova(366, 526), Fagaras->Bucharest(450, 450)]

Expanding Pitesti->Bucharest(418, 418)
Goal found!
Solution path: Arad->Sibiu->Rimnicu->Pitesti->

## **Extra Credit - explaining A\* cost optimality**

According to the textbook, an "admissible heuristic is one that never overestimates the cost to reach a goal. (An admissible heuristic is therefore optimistic.)"

For our specific case, we will be using Arad as our starting city and Bucharest as our goal, with our optimal path cost being 418, which was determined above in the A* algorithm above. 

The textbook proves this claim via proof by contradiction, so this will be sort of similar:

Assume that the optimal solution has cost 418, but A* returned a worse solution, we can say 450 for this example. 

Because of this, we can assume that there must be a node on the optimal path that was not expanded, since if every noed was expanded then the optimal goal should have been reached. 

A* uses f(n) = g(n) + h(n), and since n is on the optimal path, we can also assume that g(n) <= g*(n), which makes the new function f(n) = g*(n) + h(n)

h(n) must be <= h*(n), which means that f(n) <= g*(n) <= h*(n), therefore g*(n) + h*(n) = C*, and f(n) <= C*.

In the Romania example, any of the nodes has the optimal goal f(n) <= 418, and f(goal) = g(goal) = 450

A* should always expand the node with smallest f(n), but there also exists a node n with f(n) <= 418, and yet we have a node with f=450. This leads to a CONTRADICTION

Therefore, the assumption must be false, and A* must return the optimal (lowest-cost) solution when the heuristic is admissible.